# 02 - Data Cleaning

## Step 0: Cleaning choices

We could distinguish actual gaps in the nighttime zero data using the `H_sun` variable, but since there are neither gaps nor duplicates, imputation is not necessary.

Our data is provided in UTC. To better align with the Spanish energy market, we must convert it to CET/CEST (Spanish time), accounting for daylight saving time changes.

The main columns to be removed from the Copernicus dataset are:
- `sst` (empty)
- `cbh` (24.62% null values; secondary contribution)
- `latitude`/`longitude` (constant values; zero variance)
- `t2m` (duplicated in PVGIS)

        Important: the `t2m` variable appears in both datasets. In Copernicus, this variable represents an instantaneous value, whereas in PVGIS it is an hourly average. We retain the `t2m` variable from PVGIS as it matches the format of the other data (Gb(i), H_sun, etc.).


Variables to be removed from the PVGIS dataset:
- `Int` (always 0; provides no useful information)

Negative `ssrd` values ​​will be capped at 0 (clipped) during the cleaning process, as actual irradiance cannot be negative; such values ​​indicate a numerical artifact typical of reanalysis data near zero. `ssrd` must also be converted from $J/m^2$ to $W/m^2$.

We will convert the timestamps from UTC to CET and trim the date range to 2006–2023 for both datasets. Subsequently, we will harmonize the time windows between the two.

## Step 1: Useless data cleaning

In [227]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [228]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [229]:
from src.config import (
    COPERNICUS_DATA_FILENAME,
    PROCESSED_DATA_DIR,
    PVGIS_DATA_FILENAME,
    RAW_DATA_DIR,
)
from src.data.ingest import load_raw_cloudiness_data, load_raw_irradiance_data
from src.data.preprocess import (
    clean_cloudiness_data,
    clean_irradiance_data,
    cross_datasets,
)

irradiance_data = load_raw_irradiance_data(RAW_DATA_DIR / PVGIS_DATA_FILENAME)
cloudiness_data = load_raw_cloudiness_data(RAW_DATA_DIR / COPERNICUS_DATA_FILENAME)

### Irradiance Data

In [230]:
irradiance_data.head()

,time,P,Gb(i),Gd(i),Gr(i),H_sun,T2m,WS10m,Int
0,2005-01-01 00:30:00,0.0,0.0,0.0,0.0,0.0,12.60,3.17,0.0
1,2005-01-01 01:30:00,0.0,0.0,0.0,0.0,0.0,12.48,2.76,0.0
2,2005-01-01 02:30:00,0.0,0.0,0.0,0.0,0.0,12.38,1.66,0.0
3,2005-01-01 03:30:00,0.0,0.0,0.0,0.0,0.0,12.07,0.90,0.0
4,2005-01-01 04:30:00,0.0,0.0,0.0,0.0,0.0,11.75,0.34,0.0


In [231]:
irradiance_data = clean_irradiance_data(irradiance_data)

In [232]:

display(irradiance_data.info())
display(irradiance_data.head())

<class 'pandas.DataFrame'>
RangeIndex: 157776 entries, 8759 to 166534
Data columns (total 8 columns):
 #   Column  Non-Null Count   Dtype                        
---  ------  --------------   -----                        
 0   time    157776 non-null  datetime64[us, Europe/Madrid]
 1   P       157776 non-null  float64                      
 2   Gb(i)   157776 non-null  float64                      
 3   Gd(i)   157776 non-null  float64                      
 4   Gr(i)   157776 non-null  float64                      
 5   H_sun   157776 non-null  float64                      
 6   T2m     157776 non-null  float64                      
 7   WS10m   157776 non-null  float64                      
dtypes: datetime64[us, Europe/Madrid](1), float64(7)
memory usage: 9.6 MB


None

,time,P,Gb(i),Gd(i),Gr(i),H_sun,T2m,WS10m
8759,2006-01-01 00:00:00+01:00,0.0,0.0,0.0,0.0,0.0,12.72,6.14
8760,2006-01-01 01:00:00+01:00,0.0,0.0,0.0,0.0,0.0,12.59,7.10
8761,2006-01-01 02:00:00+01:00,0.0,0.0,0.0,0.0,0.0,12.38,7.72
8762,2006-01-01 03:00:00+01:00,0.0,0.0,0.0,0.0,0.0,12.21,7.79
8763,2006-01-01 04:00:00+01:00,0.0,0.0,0.0,0.0,0.0,12.03,7.79


### Cloudiness Data


In [233]:
cloudiness_data.head()

,valid_time,v100,v10,d2m,t2m,blh,cbh,msl,sst,skt,slhf,sp,sshf,ssrd,strd,tcc,tp,fdir,latitude,longitude
0,2006-01-01 00:00:00,0.784286,0.346283,278.47076,281.52014,474.01923,527.69104,101056.560,NaN,279.50934,-48159.0,98977.516,32718.0,0.0,1111961.20,0.983185,0.000000e+00,0.0,41.5,2.25
1,2006-01-01 01:00:00,-0.314621,-0.198929,278.65588,281.94220,594.76575,1400.01900,101037.875,NaN,280.03403,-56592.0,98955.700,65555.0,0.0,1066923.80,0.742828,4.768372e-07,0.0,41.5,2.25
2,2006-01-01 02:00:00,-1.158997,-0.526077,278.33942,281.57940,669.59320,1503.72120,101008.940,NaN,280.30255,-56844.0,98924.625,79512.0,0.0,990057.94,0.971588,4.768372e-07,0.0,41.5,2.25
3,2006-01-01 03:00:00,-2.188812,-1.077423,277.91210,281.33080,527.99915,605.32367,101013.560,NaN,280.43246,-59865.0,98927.890,51426.0,0.0,1061742.10,0.986359,9.536743e-07,0.0,41.5,2.25
4,2006-01-01 04:00:00,-2.454651,-1.188324,278.11267,281.11783,163.04854,675.10790,100950.560,NaN,279.63710,-59507.0,98864.330,30478.0,0.0,1112992.60,0.934662,1.478195e-05,0.0,41.5,2.25


In [234]:
cloudiness_data = clean_cloudiness_data(cloudiness_data)

In [235]:
display(cloudiness_data.info())
display(cloudiness_data.head())

<class 'pandas.DataFrame'>
RangeIndex: 157776 entries, 0 to 157775
Data columns (total 15 columns):
 #   Column  Non-Null Count   Dtype                        
---  ------  --------------   -----                        
 0   time    157776 non-null  datetime64[us, Europe/Madrid]
 1   v100    157776 non-null  float64                      
 2   v10     157776 non-null  float64                      
 3   d2m     157776 non-null  float64                      
 4   blh     157776 non-null  float64                      
 5   msl     157776 non-null  float64                      
 6   skt     157776 non-null  float64                      
 7   slhf    157776 non-null  float64                      
 8   sp      157776 non-null  float64                      
 9   sshf    157776 non-null  float64                      
 10  ssrd    157776 non-null  float64                      
 11  strd    157776 non-null  float64                      
 12  tcc     157776 non-null  float64                      


None

,time,v100,v10,d2m,blh,msl,skt,slhf,sp,sshf,ssrd,strd,tcc,tp,fdir
0,2006-01-01 00:00:00+01:00,0.784286,0.346283,278.47076,474.01923,101056.560,279.50934,-48159.0,98977.516,32718.0,0.0,1111961.20,0.983185,0.000000e+00,0.0
1,2006-01-01 01:00:00+01:00,-0.314621,-0.198929,278.65588,594.76575,101037.875,280.03403,-56592.0,98955.700,65555.0,0.0,1066923.80,0.742828,4.768372e-07,0.0
2,2006-01-01 02:00:00+01:00,-1.158997,-0.526077,278.33942,669.59320,101008.940,280.30255,-56844.0,98924.625,79512.0,0.0,990057.94,0.971588,4.768372e-07,0.0
3,2006-01-01 03:00:00+01:00,-2.188812,-1.077423,277.91210,527.99915,101013.560,280.43246,-59865.0,98927.890,51426.0,0.0,1061742.10,0.986359,9.536743e-07,0.0
4,2006-01-01 04:00:00+01:00,-2.454651,-1.188324,278.11267,163.04854,100950.560,279.63710,-59507.0,98864.330,30478.0,0.0,1112992.60,0.934662,1.478195e-05,0.0


## Step 2: Combining and unifying data

In [236]:
combined_dataset = cross_datasets(irradiance_data, cloudiness_data)
display(combined_dataset.head(10))
display(combined_dataset.info())
display(combined_dataset.describe())

,time,P,Gb(i),Gd(i),Gr(i),H_sun,T2m,WS10m,v100,v10,...,msl,skt,slhf,sp,sshf,ssrd,strd,tcc,tp,fdir
0,2006-01-01 00:00:00+01:00,0.0,0.00,0.00,0.00,0.00,12.72,6.14,0.784286,0.346283,...,101056.560,279.50934,-48159.0,98977.516,32718.0,0.000000,1111961.20,0.983185,0.000000e+00,0.0
1,2006-01-01 01:00:00+01:00,0.0,0.00,0.00,0.00,0.00,12.59,7.10,-0.314621,-0.198929,...,101037.875,280.03403,-56592.0,98955.700,65555.0,0.000000,1066923.80,0.742828,4.768372e-07,0.0
2,2006-01-01 02:00:00+01:00,0.0,0.00,0.00,0.00,0.00,12.38,7.72,-1.158997,-0.526077,...,101008.940,280.30255,-56844.0,98924.625,79512.0,0.000000,990057.94,0.971588,4.768372e-07,0.0
3,2006-01-01 03:00:00+01:00,0.0,0.00,0.00,0.00,0.00,12.21,7.79,-2.188812,-1.077423,...,101013.560,280.43246,-59865.0,98927.890,51426.0,0.000000,1061742.10,0.986359,9.536743e-07,0.0
4,2006-01-01 04:00:00+01:00,0.0,0.00,0.00,0.00,0.00,12.03,7.79,-2.454651,-1.188324,...,100950.560,279.63710,-59507.0,98864.330,30478.0,0.000000,1112992.60,0.934662,1.478195e-05,0.0
5,2006-01-01 05:00:00+01:00,0.0,0.00,0.00,0.00,0.00,11.59,7.10,-2.171432,-1.338730,...,100980.125,278.88763,-52147.0,98891.360,20780.0,0.000000,1100159.90,0.818939,1.811981e-05,0.0
6,2006-01-01 06:00:00+01:00,0.0,0.00,0.00,0.00,0.00,10.94,6.28,-1.936676,-1.455963,...,101040.940,278.18310,-45239.0,98948.860,16729.0,0.000000,1061337.20,0.847229,1.287460e-05,0.0
7,2006-01-01 07:00:00+01:00,0.0,0.00,0.00,0.00,0.00,10.36,5.72,-1.737228,-1.157577,...,101114.500,278.20953,-31729.0,99017.170,398.0,0.000000,1062791.00,0.810059,2.813339e-05,0.0
8,2006-01-01 08:00:00+01:00,0.1,0.00,5.76,0.15,1.18,10.04,5.38,-2.025604,-1.029419,...,101190.125,279.07530,-31915.0,99088.625,4146.0,10.382222,1050031.80,0.454071,1.349449e-04,14976.0
9,2006-01-01 09:00:00+01:00,158.3,135.84,58.72,1.90,10.05,9.84,4.76,-2.074860,-1.321457,...,101308.250,281.44778,-56640.0,99201.695,-11466.0,91.324444,1023788.00,0.340301,1.235008e-04,178560.0


<class 'pandas.DataFrame'>
RangeIndex: 157776 entries, 0 to 157775
Data columns (total 22 columns):
 #   Column  Non-Null Count   Dtype                        
---  ------  --------------   -----                        
 0   time    157776 non-null  datetime64[us, Europe/Madrid]
 1   P       157776 non-null  float64                      
 2   Gb(i)   157776 non-null  float64                      
 3   Gd(i)   157776 non-null  float64                      
 4   Gr(i)   157776 non-null  float64                      
 5   H_sun   157776 non-null  float64                      
 6   T2m     157776 non-null  float64                      
 7   WS10m   157776 non-null  float64                      
 8   v100    157776 non-null  float64                      
 9   v10     157776 non-null  float64                      
 10  d2m     157776 non-null  float64                      
 11  blh     157776 non-null  float64                      
 12  msl     157776 non-null  float64                      


None

,P,Gb(i),Gd(i),Gr(i),H_sun,T2m,WS10m,v100,v10,d2m,...,msl,skt,slhf,sp,sshf,ssrd,strd,tcc,tp,fdir
count,157776.000000,157776.000000,157776.000000,157776.000000,157776.000000,157776.000000,157776.000000,157776.000000,157776.000000,157776.000000,...,157776.000000,157776.000000,1.577760e+05,157776.000000,1.577760e+05,157776.000000,1.577760e+05,157776.000000,157776.000000,1.577760e+05
mean,179.694992,164.054867,58.142835,4.081967,15.131938,17.332716,4.034545,0.251385,0.106781,284.653697,...,101668.265317,289.550670,-2.080136e+05,99610.284165,-1.058788e+05,182.204591,1.179815e+06,0.483298,0.000070,4.567446e+05
std,250.967731,245.618875,73.013945,5.690116,20.231911,5.187768,3.194361,2.698991,2.035935,6.086565,...,695.526840,7.355503,1.966341e+05,681.622927,1.914753e+05,253.786523,1.538080e+05,0.376813,0.000321,7.131953e+05
min,0.000000,0.000000,0.000000,0.000000,-5.630000,2.490000,0.000000,-14.761261,-10.095184,260.081800,...,98125.060000,272.733550,-1.322378e+06,96117.050000,-1.158995e+06,0.000000,6.501370e+05,0.000000,0.000000,0.000000e+00
25%,0.000000,0.000000,0.000000,0.000000,0.000000,12.960000,2.140000,-1.409031,-1.308033,280.019312,...,101299.500000,283.417525,-3.137365e+05,99256.075000,-1.520345e+05,0.000000,1.056656e+06,0.089935,0.000000,0.000000e+00
50%,0.000000,0.000000,5.220000,0.140000,0.200000,16.580000,3.380000,0.131615,-0.157822,284.661270,...,101683.625000,289.119735,-1.347820e+05,99636.266000,-1.708400e+04,8.800000,1.186846e+06,0.460083,0.000000,3.392000e+03
75%,368.552500,292.022500,119.162500,7.650000,27.972500,22.060000,5.170000,2.014824,1.470325,289.869172,...,102069.810000,294.806235,-5.653975e+04,100005.095000,3.753000e+03,337.155556,1.303176e+06,0.883789,0.000007,7.424000e+05
max,911.980000,957.460000,392.710000,21.740000,71.440000,30.180000,37.790000,14.282288,9.590134,298.125920,...,104049.810000,314.045780,4.511500e+04,101906.250000,4.601390e+05,1017.795556,1.592084e+06,1.000000,0.012948,3.225664e+06


In [237]:
len(combined_dataset) == len(irradiance_data) == len(cloudiness_data)

True

Our data has been effectively cleaned and combined!

In [238]:
combined_dataset.set_index('time', inplace=True)
combined_dataset.to_csv(PROCESSED_DATA_DIR / 'combined_dataset.csv', sep=',', encoding='utf-8', index=True)

Processed data csv saved in `data/processed`